# Merge Caspase and Mitochondrial HepG2 Data

This notebook merges two datasets:
- **AID 720634**: Mitochondrial viability HepG2
- **AID 1346981**: Caspase-3/7 viability HepG2

**Steps:**
1. Load original data (skip first 5 metadata rows)
2. Remove rows with missing CID
3. Remove duplicates within each dataset
4. Combine and do final deduplication: ONE row per CID+outcome (keep highest score)

In [3]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Step 1: Load and Clean AID 720634 (Mitochondrial HepG2)

In [ ]:
# Load AID 720634 - skip first 5 metadata rows
mito_raw = pd.read_csv('Data/mitochondrial/AID_720634_datatable_viability.csv', 
                       skiprows=[1,2,3,4,5], 
                       low_memory=False)

print("AID 720634 - Mitochondrial HepG2")
print(f"Loaded shape: {mito_raw.shape}")

# Remove rows with missing CID
mito_raw['PUBCHEM_CID'] = pd.to_numeric(mito_raw['PUBCHEM_CID'], errors='coerce')
print(f"Rows with missing CID: {mito_raw['PUBCHEM_CID'].isna().sum()}")
mito_raw = mito_raw.dropna(subset=['PUBCHEM_CID'])
mito_raw['PUBCHEM_CID'] = mito_raw['PUBCHEM_CID'].astype(int)
mito_raw['PUBCHEM_ACTIVITY_SCORE'] = pd.to_numeric(mito_raw['PUBCHEM_ACTIVITY_SCORE'], errors='coerce')

print(f"After cleanup: {mito_raw.shape}")
print(f"Unique CIDs: {mito_raw['PUBCHEM_CID'].nunique()}")

AID 720634 - Mitochondrial HepG2
Loaded shape: (10486, 1764)
Rows with missing CID: 149
After cleanup: (10337, 1764)
Unique CIDs: 8099


In [32]:
# Remove duplicates within mito: keep highest score per CID+outcome
mito_sorted = mito_raw.sort_values(
    by=['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_SCORE'],
    ascending=[True, False]
)

mito_clean = mito_sorted.drop_duplicates(
    subset=['PUBCHEM_CID'],
    keep='first'
).copy()

print(f"After dedup: {mito_raw.shape} -> {mito_clean.shape}")
print(f"Removed {len(mito_raw) - len(mito_clean)} duplicate rows")

print(mito_clean.head())


# Verify: Check CID 6
print("Verification - CID 4:")
print("\nBefore dedup (combined):")
display(mito_raw[mito_raw['PUBCHEM_CID'] == 403][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']])

print("\nAfter dedup (merged):")
display(mito_clean[mito_clean['PUBCHEM_CID'] == 403][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE']])

After dedup: (10337, 1764) -> (8099, 1764)
Removed 2238 duplicate rows
      PUBCHEM_RESULT_TAG  PUBCHEM_SID  PUBCHEM_CID  \
8467                8468    144212021            4   
5600                5601    144209154            6   
6110                6111    144209664           11   
4955                4956    144208509           13   
7915                7916    144211469           33   

                  PUBCHEM_EXT_DATASOURCE_SMILES PUBCHEM_ACTIVITY_OUTCOME  \
8467                                    CC(CN)O                 Inactive   
5600  C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl                   Active   
6110                                   C(CCl)Cl                 Inactive   
4955                      C1=CC(=C(C=C1Cl)Cl)Cl                 Inactive   
7915                                   C(C=O)Cl                 Inactive   

      PUBCHEM_ACTIVITY_SCORE  \
8467                       0   
5600                      81   
6110                       0   
4955               

,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
3239,403,Inconclusive,20
3472,403,Inconclusive,20
4674,403,Active,40



After dedup (merged):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE
4674,403,Active,40


In [33]:
# Save
mito_clean.to_csv('AID_720634_cleaned.csv', index=False)

## Step 3: Standardize and Combine

In [8]:
# Define assay metadata
ASSAY_METADATA = {
    '720634': {
        'assay_name': 'AID_720634_mito_HepG2',
        'assay_type': 'mitochondrial_viability_hepg2',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': True,
        'biological_process': 'cytotoxicity',
        'endpoint_level': 'general toxicity'
    },
    '1346981': {
        'assay_name': 'AID_1346981_caspase37_HepG2',
        'assay_type': 'caspase_viability_hepg2',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': True,
        'biological_process': 'cytotoxicity',
        'endpoint_level': 'general toxicity'
    }
}

def create_std_df(df, aid):
    meta = ASSAY_METADATA[aid]
    return pd.DataFrame({
        'CID': df['PUBCHEM_CID'].values,
        'SMILES': df['PUBCHEM_EXT_DATASOURCE_SMILES'].values,
        'outcome': df['PUBCHEM_ACTIVITY_OUTCOME'].values,
        'score': df['PUBCHEM_ACTIVITY_SCORE'].values,
        'assay_name': meta['assay_name'],
        'assay_type': meta['assay_type'],
        'cell_line': meta['cell_line'],
        'source': meta['source'],
        'is_counter_screen': meta['is_counter_screen'],
        'biological_process': meta['biological_process'],
        'endpoint_level': meta['endpoint_level']
    })

mito_std = create_std_df(mito_clean, '720634')
caspase_std = create_std_df(caspase_clean, '1346981')

# Combine both datasets
combined = pd.concat([mito_std, caspase_std], ignore_index=True)
print(f"Combined: {len(combined)} rows")

Combined: 16341 rows


## Step 4: Final Deduplication - ONE row per CID+outcome (highest score)

In [9]:
# Sort by CID, outcome, score (descending)
combined_sorted = combined.sort_values(
    by=['CID', 'outcome', 'score'],
    ascending=[True, True, False]
)

# Keep only ONE row per CID + outcome (the one with highest score)
merged_df = combined_sorted.drop_duplicates(
    subset=['CID', 'outcome'],
    keep='first'
).reset_index(drop=True)

print(f"Before final dedup: {len(combined)} rows")
print(f"After final dedup: {len(merged_df)} rows")
print(f"Removed {len(combined) - len(merged_df)} duplicate CID+outcome rows")

Before final dedup: 16341 rows
After final dedup: 10431 rows
Removed 5910 duplicate CID+outcome rows


In [22]:
# Verify: Check CID 6


print("Verification - CID 4:")
print("\nBefore dedup (combined):")
display(combined[combined['CID'] == 13493][['CID', 'outcome', 'score', 'assay_name']])

print("\nAfter dedup (merged):")
display(merged_df[merged_df['CID'] == 13493][['CID', 'outcome', 'score', 'assay_name']])

KeyError: 'CID'

In [11]:
# Summary
print("="*60)
print("MERGED DATASET SUMMARY")
print("="*60)
print(f"Total records: {len(merged_df):,}")
print(f"Unique CIDs: {merged_df['CID'].nunique():,}")

print("\nBy assay_name (source of highest score):")
print(merged_df['assay_name'].value_counts())

print("\nBy outcome:")
for outcome in merged_df['outcome'].value_counts().index:
    count = merged_df['outcome'].value_counts()[outcome]
    pct = count / len(merged_df) * 100
    print(f"  {outcome}: {count:,} ({pct:.1f}%)")

MERGED DATASET SUMMARY
Total records: 10,431
Unique CIDs: 8,734

By assay_name (source of highest score):
assay_name
AID_720634_mito_HepG2          8266
AID_1346981_caspase37_HepG2    2165
Name: count, dtype: int64

By outcome:
  Inactive: 8,200 (78.6%)
  Inconclusive: 1,477 (14.2%)
  Active: 754 (7.2%)


In [12]:
# Display sample
print("First 10 rows:")
display(merged_df.head(10))

First 10 rows:


,CID,SMILES,outcome,score,assay_name,assay_type,cell_line,source,is_counter_screen,biological_process,endpoint_level
0,4,CC(CN)O,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
1,6,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,Active,81,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
2,11,C(CCl)Cl,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
3,13,C1=CC(=C(C=C1Cl)Cl)Cl,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
4,33,C(C=O)Cl,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
5,34,C(CCl)O,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
6,51,C(CC(=O)O)C(=O)C(=O)O,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
7,66,C1=CC(=C(C=C1Cl)O)Cl,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
8,66,C1=CC(=C(C=C1Cl)O)Cl,Inconclusive,20,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity
9,72,C1=CC(=C(C=C1C(=O)O)O)O,Inactive,0,AID_720634_mito_HepG2,mitochondrial_viability_hepg2,HepG2,Tox21,True,cytotoxicity,general toxicity


In [13]:
# Save
merged_df.to_csv('merged_caspase_mito_hepg2.csv', index=False)
print(f"Saved to: merged_caspase_mito_hepg2.csv")
print(f"Shape: {merged_df.shape}")
print(f"Columns: {list(merged_df.columns)}")

Saved to: merged_caspase_mito_hepg2.csv
Shape: (10431, 11)
Columns: ['CID', 'SMILES', 'outcome', 'score', 'assay_name', 'assay_type', 'cell_line', 'source', 'is_counter_screen', 'biological_process', 'endpoint_level']
